# Tutorial 3 — HCHR mode: AF Lep b

**What you'll learn:** How to run a High-Contrast High-Resolution (HCHR) atmosphere
retrieval with ForMoSA v2.0. HCHR combines starlight suppression (high-contrast)
with molecular line spectroscopy (high-resolution) to characterise companions
that are otherwise buried in stellar speckles.

**Target:** AF Leporis b — a young, directly-imaged companion detected with
VLT/HiRISE (SPHERE + CRIRES+) at R ≈ 140,000 in the K-band.

**Grid:** Exo-REM cloudless (Charnay et al. 2018) — a self-consistent cloudy/cloudless
model grid tuned for young directly-imaged companions.

**Parameters fitted:** Teff, log g, r, d, rv, vsini

> **Note:** The data for this tutorial is not yet publicly available.
> The data download cell is a placeholder that will be updated once the data is published.
> All other cells are complete and correct.

**References:** Zhang et al. (2023); Petrus et al. (2021) for the HCHR method


## Section 0: Setup

In [ ]:
# Cell A: Environment check
import sys
try:
    import ForMoSA
    print(f"ForMoSA {ForMoSA.__version__} — OK")
except ImportError:
    raise ImportError("pip install ForMoSA && conda install dask netCDF4 bottleneck")
print(f"Python {sys.version.split()[0]}")


In [ ]:
# Cell B: Workspace setup
from pathlib import Path
TUTORIAL_DIR = Path(".").resolve()
for d in ["data", "adapted_grid", "results", "grid"]:
    (TUTORIAL_DIR / d).mkdir(exist_ok=True)
print(f"Working directory: {TUTORIAL_DIR}")


In [ ]:
# Cell C: Data download + validation
# TODO: Replace the URL below once AF Lep b HiRISE data is published.
# The expected file is a FITS table with these extensions:
#   WAV        — wavelength array (µm)
#   WAVE_UNIT  — wavelength unit string
#   FLX        — planet flux (speckle-subtracted, W/m²/µm)
#   ERR        — per-channel noise
#   RES        — spectral resolution R = λ/Δλ per wavelength point
#   STAR_FLUX  — stellar speckle reference spectrum (triggers hc_mode automatically)

from astropy.io import fits
from pathlib import Path

DATA_FILE = TUTORIAL_DIR / "data" / "AFLepb_HiRISE.fits"
# DATA_URL = "https://github.com/exoAtmospheres/ForMoSA/releases/download/tutorial-data-v1/AFLepb_HiRISE.fits"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Data file not found: {DATA_FILE}\n"
        "The AF Lep b HiRISE dataset is not yet publicly available.\n"
        "Once published, uncomment the DATA_URL line above and add the download code."
    )

REQUIRED = {"WAV", "WAVE_UNIT", "FLX", "ERR", "RES", "STAR_FLUX"}
with fits.open(DATA_FILE) as hdul:
    found = {ext.name for ext in hdul[1:]}

missing = REQUIRED - {k.upper() for k in found}
if missing:
    raise RuntimeError(f"Missing FITS extensions: {missing}")

print("FITS extensions (required marked ✓):")
for name in sorted(found):
    mark = "✓" if name.upper() in REQUIRED else " "
    print(f"  {mark} {name}")
print("\nNote: STAR_FLUX triggers hc_mode=True automatically — no manual flag needed.")


In [ ]:
# Cell D: Grid download
# TODO: Replace with Exo-REM grid URL once available on GitHub Releases.
import urllib.request
from pathlib import Path

GRID_FILE = TUTORIAL_DIR / "grid" / "Exo-REM_cloudless.nc"
# GRID_URL = "https://github.com/exoAtmospheres/ForMoSA/releases/download/tutorial-data-v1/Exo-REM_cloudless.nc"

if not GRID_FILE.exists():
    raise FileNotFoundError(
        f"Grid file not found: {GRID_FILE}\n"
        "The Exo-REM grid download URL will be added here once the release is prepared.\n"
        "In the meantime, download the grid from https://lesia.obspm.fr/exorem/ and set GRID_FILE."
    )

import xarray as xr
ds = xr.open_dataset(GRID_FILE, decode_cf=False)
print(f"Grid: {dict(ds.sizes)}")
par_names = ds.attrs.get("par", [])
par_units = ds.attrs.get("unit", [])
for i, key in enumerate(["par1","par2","par3","par4"]):
    if key in ds.coords:
        vals = ds[key].values
        name = par_names[i] if i < len(par_names) else key
        unit = par_units[i] if i < len(par_units) else ""
        print(f"  {name} {unit}: {vals[0]:.1f} → {vals[-1]:.1f}  ({len(vals)} points)")


## Section 1: The science

### What is HCHR?

High-Contrast (HC) + High-Resolution (HR) spectroscopy is a technique that
combines the spatial suppression of starlight from adaptive optics / coronagraphy
with the Doppler shift of molecular lines to detect companions lost in speckle noise.

At R ≈ 140,000, individual CO and OH rotational lines are fully resolved.
The stellar and planetary spectra are shifted relative to each other by their
radial velocity difference — typically tens of km/s — allowing them to be separated
even without coronagraphic starlight suppression.

### The STAR_FLUX extension

In HCHR mode, the `STAR_FLUX` FITS extension carries the stellar speckle reference
spectrum (e.g., from a reference star or from the science star itself). ForMoSA
uses it to model and remove the stellar contamination in the planet spectrum.

**ForMoSA detects HCHR mode automatically:** if a `STAR_FLUX` column is present
in your FITS file, `hc_mode=True` is set, which:
- **Disables continuum removal** from models (the speckle model handles broadband shape)
- **Applies `_hc_modeling`** instead of analytical scaling
- **Disables the covariance matrix** (not yet implemented for HCHR)

### FITS extensions — all required for HCHR

| Extension | Type | Description |
|-----------|------|-------------|
| `WAV` | 1D float | Wavelength array (µm) |
| `WAVE_UNIT` | string | Wavelength unit (`um`, `nm`, `AA`) |
| `FLX` | 1D float | Planet flux (speckle-subtracted) |
| `ERR` | 1D float | Per-channel noise (1σ) |
| `RES` | 1D float | Spectral resolution R = λ/Δλ |
| `STAR_FLUX` | 2D float | Stellar speckle reference (one column per nod/epoch) |

### AF Lep b

AF Leporis b is a young (~24 Myr) super-Jupiter companion at ~27 pc detected
by direct imaging and characterised with VLT/HiRISE. At R ≈ 140,000, HiRISE
can measure both the radial velocity and the projected rotation rate (vsini)
of the companion by resolving individual CO lines.


## Section 2: Inspect the data

In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

with fits.open(DATA_FILE) as hdul:
    wav      = hdul["WAV"].data.astype(float)
    flx      = hdul["FLX"].data.astype(float)
    err      = hdul["ERR"].data.astype(float)
    res      = hdul["RES"].data.astype(float)
    # STAR_FLUX may be 2D: (n_wavelengths, n_nods)
    star_flx = hdul["STAR_FLUX"].data.astype(float)

print(f"Wavelength range : {wav.min():.4f} – {wav.max():.4f} µm")
print(f"Resolution R     : {res.mean():.0f} ± {res.std():.0f}")
print(f"STAR_FLUX shape  : {star_flx.shape}  (wavelengths × nod positions)")

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(wav, flx, lw=0.6, color="#2E86AB", label="Planet flux")
axes[0].fill_between(wav, flx-err, flx+err, alpha=0.3, color="#2E86AB")
axes[0].set_ylabel("Flux"); axes[0].legend()
axes[1].plot(wav, star_flx[:, 0] if star_flx.ndim == 2 else star_flx,
             lw=0.6, color="#E84855", label="Star flux (nod 0)")
axes[1].set_ylabel("Star flux"); axes[1].legend()
axes[2].plot(wav, res, lw=0.6, color="#555")
axes[2].set_ylabel(r"Resolution $R$")
axes[2].set_xlabel(r"Wavelength (µm)")
plt.tight_layout(); plt.show()


## Section 3: Configure the analysis

In [ ]:
from ForMoSA.config.global_config import ConfigPath, ConfigAdapt, ConfigInversion, ConfigParameters

config_path = ConfigPath(
    observation_path=[str(DATA_FILE)],
    adapt_store_path=str(TUTORIAL_DIR / "adapted_grid"),
    result_path=str(TUTORIAL_DIR / "results"),
    model_path=str(GRID_FILE),
)

# hc_mode is detected automatically from STAR_FLUX — no flag needed here.
# res_cont is ignored in hc_mode (continuum removal is disabled automatically).
config_adapt = ConfigAdapt()

config_inversion = ConfigInversion(
    wav_fit=["2.28, 2.36"],   # CO bandhead region in K-band (µm)
    ns_algo="nestle",
    npoints=300,
    logL_type=["chi2"],
    # HCHR-specific bounds for the least-squares speckle model:
    hc_lower_bounds_lsq=[-0.5],   # lower bound on speckle coefficient
    hc_higher_bounds_lsq=[2.0],   # upper bound on speckle coefficient
)

config_params = ConfigParameters(
    par1=["uniform", "800", "2000"],    # Teff (K) — Exo-REM range
    par2=["uniform", "3.0", "5.5"],    # log g (dex)
    r=["uniform", "0.5", "2.0"],       # radius (R_Jupiter)
    d=["constant", "26.8"],            # distance (pc) — Gaia DR3
    rv=["uniform", "-100", "100"],     # radial velocity (km/s)
    vsini=["uniform", "0", "60"],      # projected rotation (km/s)
    # vsini is physically meaningful here: HiRISE R≈140,000 resolves rotational broadening
)

print("HCHR configuration:")
print(f"  hc_mode   : detected automatically via STAR_FLUX extension")
print(f"  wav_fit   : {config_inversion.wav_fit[0]} µm")
print(f"  Free      : par1 (Teff), par2 (log g), r, rv, vsini")
print(f"  Fixed     : d = 26.8 pc")


## Section 4: Adapt the grid

In [ ]:
from ForMoSA import Analysis

adapted = False

analysis = Analysis(config_path, adapted=adapted, fitted=False)
if not adapted:
    print("Adapting grid to HCHR observation (convolution to R≈140,000)...")
    analysis.adapt(config_adapt, config_inversion)
    print("Done.")


## Section 5: Run the nested sampling fit

In [ ]:
from ForMoSA.config.global_config import Config_NS

config_ns = Config_NS()
print(f"Running {config_inversion.ns_algo} with {config_inversion.npoints} live points...")
analysis.nested_sampling(config_params, config_adapt, config_inversion, config_NS=config_ns)
print("Fit complete.")


## Section 6: Results

In [ ]:
analysis.plot(analysis.ns.results)
print(analysis.ns.results.summary(sigma=1))


## Section 7: INI file alternative

In [ ]:
from ForMoSA.config.global_config import ConfigGenerator
generator = ConfigGenerator()
generator.save(str(TUTORIAL_DIR), "config.ini")
print(f"Template: {TUTORIAL_DIR / 'config.ini'}")
print("Edit and load with ConfigLoader — see Tutorial 1 Section 7 for the full pattern.")


## Section 8: Next steps

- **Tutorial 4 — MOSAIC mode (β Pic b):** Combine a spectrum and photometry
  in a single simultaneous fit with per-instrument intercalibration.
- **Tutorial 6 — Cluster / MPI deployment:** Run this fit with PyMultiNest
  on an HPC cluster for faster convergence with more live points.
